In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Trajectory attribution uncertainty follow-up V1

This fixed Run-all reuses the six already-audited post-codec received.rgb8 inputs for C1/C2 A_M05, OFF and B_M05 and the exact frozen attribution rule from run 20261008T085244020205Z. It does not generate video, write a new watermark, change strength, invoke a codec, or introduce a new temporal attack.

The preregistered roster has 22 queries over 20 physical observations. Ten PRIMARY_PROBE queries—the same ten for both coverage questions—use A_M05/K0 with SHORT89 starts 0/46/92 or internal H1 b2 deletions k44/k132. Four edge-deletion ALIAS_CONTROL queries k1/k176 and eight existing CROP177 b2 REGRESSION_CONTROL queries are excluded from uncertainty coverage. Same-action path aliases never count as action ambiguity.

Valid sync-unreliability coverage requires complete evidence, M above the frozen threshold, and either more than one top action or one action with local m at or below the frozen threshold. Valid weak-identity coverage requires qualified unique sync, the correct complete action, complete payload evidence, exact I=0, all 32 signed integer vote differences nonnegative and at least one tied bit. Since every frozen tau_I is zero, no positive interval above zero is labeled weak. Technical, missing, interrupted or unsupported UNCERTAIN rows remain unresolved fixed slots and never count as method coverage.

The eight regression controls retain two prior positive and six prior negative decision/reason expectations. Across all 22 rows, a positive ACCEPT with a wrong action or any negative ACCEPT is a false claim; a correct positive probe ACCEPT is simply a non-trigger. No observed scientific trigger is reported as NOT_OBSERVED_FIXED_ROSTER; inputs, thresholds and samples are never replaced or retried. This notebook is a real saved-input receiver validation for the user to run. Static/CPU/fake release checks do not supply its scientific result.

UNPUBLISHED DRAFT: SOURCE_SHA=None; publish reviewed source then bind before Run all.

In [ ]:
SOURCE_SHA = None
FIXED = {'sources': 2, 'queries': 22, 'observations': 20, 'primary_queries': 10, 'alias_controls': 4, 'regression_controls': 8, 'positive_queries': 16, 'negative_queries': 6, 'sync_candidates': 11902, 'framewise_frames': 3012, 'framewise_batch8': 394, 'max_payload_reads': 22, 'max_votes': 802560, 'max_time_bit_rows': 26752, 'max_bit_rows': 704, 'sync_coverage_denominator': 10, 'weak_identity_coverage_denominator': 10}
from pathlib import Path
import datetime,hashlib,json,os,signal,subprocess,sys,time,traceback
if SOURCE_SHA is None:
    raise RuntimeError('UNPUBLISHED_DRAFT: publish reviewed source and rebuild with its immutable SHA before Run all')
if not isinstance(SOURCE_SHA,str) or len(SOURCE_SHA)!=40:
    raise RuntimeError('immutable 40-character source SHA required')
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Trajectory-Attribution-Uncertainty-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_OUTPUT=OUTPUT/'fixed_reference'
REPO=Path('/content/SC-SSTW-TRAJECTORY-ATTRIBUTION-UNCERTAINTY-V1-'+STAMP)
PYTHON=sys.executable
def write_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(value,indent=2)+'\n');os.replace(tmp,path)
def failed(stage,exc):
    captured_traceback=''.join(traceback.format_exception(type(exc),exc,exc.__traceback__))
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=f'{type(exc).__name__}: {exc}',traceback=captured_traceback,fixed_denominator=FIXED))
def logged(command,stage,cwd=None,check=True):
    child=None;code=None;primary=None;primary_tb=None;cleanup_errors=[]
    def retain_cleanup_error(label,exc):
        nonlocal primary,primary_tb
        if primary is None and not isinstance(exc,Exception):
            primary=exc;primary_tb=exc.__traceback__
        else:
            cleanup_errors.append(label+': '+repr(exc))
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            popen_group={'start_new_session':True} if os.name=='posix' else {}
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,**popen_group)
            try:
                for line in child.stdout:
                    print(line,end='',flush=True);log.write(line);log.flush()
                code=child.wait()
            except BaseException as exc:
                primary=exc;primary_tb=exc.__traceback__
            finally:
                if child is not None:
                    if os.name=='posix':
                        pgid=child.pid
                        try:os.killpg(pgid,signal.SIGTERM)
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('group SIGTERM',exc)
                        try:
                            deadline=time.monotonic()+5.0
                            while time.monotonic()<deadline:
                                try:os.killpg(pgid,0)
                                except ProcessLookupError:break
                                except BaseException as exc:
                                    retain_cleanup_error('group probe',exc);break
                                time.sleep(0.05)
                        except BaseException as exc:
                            retain_cleanup_error('group TERM grace',exc)
                        finally:
                            try:os.killpg(pgid,signal.SIGKILL)
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('group SIGKILL',exc)
                    elif child.poll() is None:
                        try:child.terminate()
                        except ProcessLookupError:pass
                        except BaseException as exc:retain_cleanup_error('terminate',exc)
                    try:
                        try:code=child.wait(timeout=10)
                        except subprocess.TimeoutExpired:
                            try:child.kill()
                            except ProcessLookupError:pass
                            except BaseException as exc:retain_cleanup_error('kill',exc)
                            try:code=child.wait(timeout=10)
                            except BaseException as exc:retain_cleanup_error('wait after kill',exc)
                        except BaseException as exc:retain_cleanup_error('wait',exc)
                    finally:
                        if child.stdout is not None:
                            try:child.stdout.close()
                            except BaseException as exc:retain_cleanup_error('stdout close',exc)
            if primary is None and not cleanup_errors:
                log.write('EXIT '+str(code)+'\n');log.flush()
    except BaseException as wrapper_error:
        if primary is None:
            primary=wrapper_error;primary_tb=wrapper_error.__traceback__
        elif wrapper_error is not primary:
            cleanup_errors.append('log wrapper/close: '+repr(wrapper_error))
    if primary is None and cleanup_errors:
        primary=RuntimeError('process cleanup failed: '+'; '.join(cleanup_errors))
        primary_tb=primary.__traceback__
    if primary is None and check and code:
        primary=subprocess.CalledProcessError(code,command)
        primary_tb=primary.__traceback__
    if primary is not None:
        if cleanup_errors and hasattr(primary,'add_note'):
            primary.add_note('secondary cleanup errors: '+'; '.join(cleanup_errors))
        try:failed(stage,primary)
        except BaseException as record_error:
            if hasattr(primary,'add_note'):primary.add_note('failure record error: '+repr(record_error))
        raise primary.with_traceback(primary_tb)
    return code
write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator=FIXED))


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    probe="import importlib.metadata as m; pins={'torch': '2.11.0', 'diffusers': '0.39.0', 'transformers': '4.57.6', 'numpy': '2.1.3', 'accelerate': '1.15.0', 'safetensors': '0.8.0', 'huggingface-hub': '0.36.2', 'tokenizers': '0.22.2', 'sentencepiece': '0.2.2', 'ftfy': '6.3.1'}; actual={k:m.version(k) for k in pins}; assert all(actual[k].split('+')[0]==v for k,v in pins.items()),actual; from diffusers import AutoencoderKL,AutoencoderKLWan; import torch,sentencepiece,ftfy; print(actual)"
    if logged([PYTHON,'-c',probe],'DEPENDENCY_PROBE',check=False):
        logged([PYTHON,'-m','pip','install','torch==2.11.0','diffusers==0.39.0','transformers==4.57.6','numpy==2.1.3','accelerate==1.15.0','safetensors==0.8.0','huggingface-hub==0.36.2','tokenizers==0.22.2','sentencepiece==0.2.2','ftfy==6.3.1'],'PINNED_REPAIR')
        logged([PYTHON,'-c',probe],'DEPENDENCY_REPROBE')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    write_json(OUTPUT/'setup_receipt.json',dict(status='SETUP_COMPLETE',source_sha=actual,fixed_denominator=FIXED,dependency_check_returncode=dependency_code))
except Exception as exc:
    try:failed('SOURCE_OR_ENVIRONMENT',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
CONFIG_PATH=REPO/'experiments/wan_state_clock/configs/video_trajectory_attribution_uncertainty_v1.json'
CONFIG_SHA=hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
configuration=json.loads(CONFIG_PATH.read_text())
if configuration['name']!='video_trajectory_attribution_uncertainty_v1' or configuration['fixed_denominator']!=FIXED:
    raise RuntimeError('fixed configuration mismatch')
command=[PYTHON,'-u','-m','experiments.wan_state_clock.video_trajectory_attribution_uncertainty_v1_run','--config',str(CONFIG_PATH),'--output',str(RUN_OUTPUT)]
try:
    returncode=logged(command,'FIXED_SAVED_RGB_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; fixed 22-query denominator retained')
except Exception as exc:
    try:failed('RUNNER',exc)
    except BaseException as record_error:
        if hasattr(exc,'add_note'):exc.add_note('failure record error: '+repr(record_error))
    raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['config_sha256']!=CONFIG_SHA or result['fixed_denominator']!=FIXED:
    raise RuntimeError('result source/config/denominator identity mismatch')
print('Runtime:',result['status'],'Stage:',result['stage'])
print('Coverage denominator: the same 10 PRIMARY_PROBE queries are evaluated for both branches; they are not 20 independent samples.')
for branch,row in result['coverage'].items():
    if isinstance(row,dict) and 'denominator' in row:
        print('Coverage:',branch,'denominator=',row['denominator'],'actual_trigger=',row['actual_trigger'],'valid_coverage=',row['valid_coverage'],'technical=',row['technical'],'status=',row['status'])
print('Probe focus:',result['coverage']['probe_focus'],'Excluded alias/control:',result['coverage']['alias_controls_excluded'],result['coverage']['regression_controls_excluded'])
print('Regression controls:',result['controls'])
print('False attribution across all 22:',result['false_attribution'])
for query_id,row in sorted(result['queries'].items()):
    post=row['posthoc'];sync=row['sync'];identity=row['identity'];decision=row['decision']
    print(query_id,post['source'],post['condition'],post['key'],post['view'],post['parameters'],post['role'],post.get('focus'),
          decision['decision'],decision['reason'],'M=',sync.get('M'),'m=',sync.get('m'),'actions=',sync.get('top_action_count'),
          'I=',identity.get('I'),'exact32=',identity.get('exact32'),'action_correct=',post.get('action_map_correct'),
          'sync_coverage=',post['valid_sync_uncertainty_coverage'],'weak_coverage=',post['valid_weak_identity_coverage'],
          'false_claim=',post['false_claim'],'technical_complete=',post['technical_complete'])
print('Failures:',result['failures'])
print('Evidence ceiling:',result['evidence_ceiling'])
print('Result:',RESULT_PATH)
